# Mini-Project 2: Uganda Tax Engine (URA Compliance & OOP Design)

## Overview
This notebook demonstrates an extensible, testable Object-Oriented tax calculation engine built for the Uganda Revenue Authority (URA) tax rules. 
It features:
- A custom `Money` value object utilizing strict `decimal.Decimal` precision (avoiding floating-point inaccuracies).
- An abstract `Tax` base class with concrete implementations for **PAYE**, **NSSF**, **VAT**, and **Corporate Income Tax (CIT)**.
- A comprehensive **Payroll Pipeline** managing the sequence of statutory deductions.
- Custom exception handling (`TaxError`, `InvalidIncomeError`).

In [2]:
import sys
import os

# Ensure the parent directory is in the path so we can import from src/
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from src.tax_engine import (
    Money, TaxContext, PayrollPipeline, 
    PAYECalculator, NSSFCalculator, VATCalculator, CorporateIncomeTaxCalculator,
    InvalidIncomeError
)

print("Tax engine modules successfully imported!")

Tax engine modules successfully imported!


In [3]:
# Demonstrating explicit rounding and float restriction
try:
    unsafe_money = Money(100.50) # Will raise TypeError
except TypeError as e:
    print(f"Caught expected error: {e}")

# Correct usage with Decimal/String/Int
m1 = Money("1500000.556")
m2 = Money(500000)
print(f"Money 1 (Quantized): {m1}")
print(f"Money 2: {m2}")
print(f"Addition: {m1 + m2}")

Caught expected error: Float is unsafe for financial computations. Use Decimal, int, or str.
Money 1 (Quantized): Money(UGX 1,500,000.56)
Money 2: Money(UGX 500,000.00)
Addition: Money(UGX 2,000,000.56)


In [4]:
paye_calc = PAYECalculator()

# Test various salary brackets including the 10,000,000 UGX surcharge threshold
test_salaries = [300000, 400000, 450000, 5000000, 12000000]

print("--- PAYE Progressive Calculation Demo ---")
for sal in test_salaries:
    res = paye_calc.calculate(TaxContext(gross_income=Money(sal), is_resident=True))
    print(f"Gross: UGX {sal:,.2f} --> PAYE: {res.tax_amount}")

--- PAYE Progressive Calculation Demo ---
Gross: UGX 300,000.00 --> PAYE: Money(UGX 0.00)
Gross: UGX 400,000.00 --> PAYE: Money(UGX 13,000.00)
Gross: UGX 450,000.00 --> PAYE: Money(UGX 25,000.00)
Gross: UGX 5,000,000.00 --> PAYE: Money(UGX 1,388,250.00)
Gross: UGX 12,000,000.00 --> PAYE: Money(UGX 3,688,250.00)


In [5]:
# Initialize payroll pipeline with Local Service Tax (LST) of UGX 10,000
pipeline = PayrollPipeline(lst_amount=10000)

payroll_result = pipeline.process(
    gross_income=Money(4000000), 
    allowances=Money(500000), 
    is_resident=True
)

print("--- Monthly Payroll Breakdown ---")
for key, value in payroll_result.items():
    print(f"{key.replace('_', ' ').title()}: {value}")

--- Monthly Payroll Breakdown ---
Gross Income: Money(UGX 4,000,000.00)
Allowances: Money(UGX 500,000.00)
Employee Nssf: Money(UGX 200,000.00)
Local Service Tax: Money(UGX 10,000.00)
Taxable Income For Paye: Money(UGX 4,290,000.00)
Paye: Money(UGX 1,175,250.00)
Net Pay: Money(UGX 3,114,750.00)


In [6]:
# VAT calculation on sales vs allowable purchases
vat_calc = VATCalculator()
vat_res = vat_calc.calculate(TaxContext(revenue=Money(25000000), allowable_deductions=Money(10000000)))
print(f"VAT Payable: {vat_res.tax_amount} | Breakdown: {vat_res.breakdown}")

# Corporate Income Tax (CIT) at 30%
cit_calc = CorporateIncomeTaxCalculator()
cit_res = cit_calc.calculate(TaxContext(revenue=Money(100000000), allowable_deductions=Money(60000000)))
print(f"Corporate Income Tax: {cit_res.tax_amount} | Breakdown: {cit_res.breakdown}")

VAT Payable: Money(UGX 2,700,000.00) | Breakdown: {'output_vat': Money(UGX 4,500,000.00), 'input_vat': Money(UGX 1,800,000.00)}
Corporate Income Tax: Money(UGX 12,000,000.00) | Breakdown: {'chargeable_income': Money(UGX 40,000,000.00)}


## Discussion Questions

### 1. Why is float unsafe for tax computation? Show a concrete failing example.
Floating-point numbers in Python use binary representation (IEEE 754), which cannot precisely represent certain decimal fractions (like `0.1` or `0.2`). 
**Failing Example:**
\`\`\`python
# Floating point inaccuracy
print(0.1 + 0.2)  # Output: 0.30000000000000004
print(0.1 + 0.2 == 0.3)  # Output: False
\`\`\`
In a tax engine managing millions of transactions or continuous payroll runs, these microscopic rounding errors accumulate into significant financial discrepancies and compliance violations. Using `decimal.Decimal` guarantees exact base-10 arithmetic.

### 2. Where would inheritance be justified here, and where did you deliberately avoid it?
- **Justified (Inheritance):** Inheritance is used for the abstract base class `Tax` with the abstract method `calculate(self, ctx: TaxContext) -> TaxResult`. This enforces a strict, uniform interface across all tax domains (`PAYECalculator`, `NSSFCalculator`, `VATCalculator`, `CorporateIncomeTaxCalculator`), ensuring polymorphism.
- **Avoided (Composition):** Inheritance was deliberately avoided in the `PayrollPipeline`. Instead of inheriting from multiple tax types, the pipeline uses **composition**—it instantiates and orchestrates the individual calculators (`NSSFCalculator`, `PAYECalculator`) internally to execute the sequential order of payroll deductions cleanly.